In [5]:
import re


def clean_text_for_excel(text):
    """Clean text to be safe for Excel by removing illegal characters"""
    if not text:
        return ""
    
    # Remove control characters except tab (9), newline (10), carriage return (13)
    cleaned = ''.join(char for char in text if ord(char) >= 32 or ord(char) in [9, 10, 13])
    
    # Remove any remaining problematic characters that might cause XML issues
    # Remove characters that are invalid in XML 1.0
    cleaned = re.sub(r'[\x00-\x08\x0B-\x0C\x0E-\x1F\x7F-\x84\x86-\x9F]', '', cleaned)
    
    # Replace some common problematic characters
    replacements = {
        '"': "'",  # Replace smart quotes
        '"': "'",
        ''': "'",
        ''': "'",
        '–': "-",  # Replace en dash
        '—': "-",  # Replace em dash
        '…': "...", # Replace ellipsis
    }
    
    for old, new in replacements.items():
        cleaned = cleaned.replace(old, new)
    
    return cleaned


In [6]:
from src import excel, epic, utils
import time
import sys
import pandas as pd
import re

INPUT_FILE = "data/avm/avm_filled.xlsx"
OUTPUT_FILE = "data/avm/avm_filled.xlsx"

df = pd.read_excel(OUTPUT_FILE)


print(f"--- Starting Main Processing Loop ---")

for i, row in df.iterrows():
    print(f"\n======= Processing Patient Iteration {i+1}/500 =======")

    if not pd.isna(df.at[i, "docs"]):
        print(f"Patient {i+1} already checked. Skipping...")
        continue

    try:
        mrn = str(df.at[i, "Patient ID"])
        patient_found = epic.find_patient_clipboard(mrn)

        if not patient_found:
            print(f"Patient {i+1} with MRN {mrn} not found. Skipping...")
            df.at[i, "docs"] = "Patient not found"
            df.at[i, "checked"] = -1

            continue
        imaging_viewed = epic.view_imaging()

        imaging_coords = utils.group_locations(epic.find_imaging_icons())

        if len(imaging_coords) == 0:
            print(f"No imaging found for patient {i+1}. Marking as no imaging.")
            df.at[i, "docs"] = "No imaging found"
            df.at[i, "checked"] = 1
            epic.close_patient()
            continue

        collated_imaging = ""
        for coord in imaging_coords:
            epic.view_note_details(coord)
            time.sleep(1)  # Wait for the imaging details to load
            imaging_contents = str(epic.copy_note_contents())
            epic.close_note_details()
            utils.clear_clipboard()
            epic.scroll_to_top()
            collated_imaging += imaging_contents + "\n---\n"
            imaging_contents = ""
            time.sleep(0.5)  # Brief pause before next imaging

        # Clean the collated imaging to remove illegal characters for Excel
        cleaned_imaging = clean_text_for_excel(collated_imaging.strip())

        # Store the cleaned imaging and mark as checked
        df.at[i, "docs"] = cleaned_imaging
        df.at[i, "checked"] = 1
        
        # Save to Excel with error handling
        try:
            df.to_excel(OUTPUT_FILE, index=False)
            print(f"Successfully saved data for patient {i+1}")
        except Exception as save_error:
            print(f"Error saving to Excel for patient {i+1}: {save_error}")
            # Try to save a backup version with even more aggressive cleaning
            df.at[i, "docs"] = re.sub(r'[^\x20-\x7E\n\r\t]', '', cleaned_imaging)  # Only ASCII printable + whitespace
            df.to_excel(OUTPUT_FILE, index=False)
            print(f"Saved with aggressive character cleaning for patient {i+1}")
        
        epic.close_patient()

    except Exception as e:
        print(f"\n!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!", file=sys.stderr)
        print(f"!!! EXCEPTION in iteration {i+1}: {e}", file=sys.stderr)
        print(f"!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!", file=sys.stderr)
        print("Stopping main loop due to error.")
        raise e
        break

print("\n--- Main Processing Loop Finished or Stopped ---")

--- Starting Main Processing Loop ---

======= Processing Patient Iteration 1/500 =======
Patient 1 already checked. Skipping...

======= Processing Patient Iteration 2/500 =======
Patient 2 already checked. Skipping...

======= Processing Patient Iteration 3/500 =======
Patient 3 already checked. Skipping...

======= Processing Patient Iteration 4/500 =======
Patient 4 already checked. Skipping...

======= Processing Patient Iteration 5/500 =======
Patient 5 already checked. Skipping...

======= Processing Patient Iteration 6/500 =======
Patient 6 already checked. Skipping...

======= Processing Patient Iteration 7/500 =======
Patient 7 already checked. Skipping...

======= Processing Patient Iteration 8/500 =======
Patient 8 already checked. Skipping...

======= Processing Patient Iteration 9/500 =======
Patient 9 already checked. Skipping...

======= Processing Patient Iteration 10/500 =======
Patient 10 already checked. Skipping...

======= Processing Patient Iteration 11/500 =====

In [5]:
import pyautogui


len(list(pyautogui.locateAllOnScreen("assets/imaging_icon.png", confidence=0.80)))

28

In [1]:
import pyautogui
from src import utils, epic

len(utils.group_locations(list(pyautogui.locateAllOnScreen("assets/imaging_icon.png", confidence=0.90))))

4

In [3]:
epic.copy_note_contents()

'\xa0\nPACS Images\n\n\xa0Show images for MR Stereo Gamma Knife Post Contrast\nVNA (Business Continuity) Images\n\n\xa0Show images for MR Stereo Gamma Knife Post Contrast\n\n\nStudy Result\n\nNarrative & Impression\nStudy Date: 27/5/22  Accession No. RRV2575781\nExam Name: MR Stereo Gamma Knife Post Contrast \n\xa0\nClinical Indications: \nLarge fronto-opercular AVM -planned for second staged Gamma KNife treatment. Had first stage in March 2022. The nned for having a new angio to be decided on the day by the INR, after the MRI\n\xa0\nReport:\nImages provided for gamma knife planning.\n\xa0\nSome evidence of response to the first treatment, lower half of left AVM target slightly adjusted. No need for repeat angiography.\n\xa0\n\xa0\nPeter Cowley\nConsultant Radiologist\nGMC : 4341765\nUniversity College London Hospitals NHS Foundation Trust\nuclh.referrals.neurorad@nhs.net\n\xa0\nScans on Order 145011123\n\n\nDocument type: Patient Safety Checklist - Media type: Scan Date: 27/5/2022 09:

In [10]:
PROMPT = """Analyze the provided radiology reports. Your task is to produce a single JSON object with no other text.

The JSON must have two keys:
1.  `"tectal/perimesencephalic/brainstem AVM"`: The value must be a boolean (`true` or `false`). Set it to `true` only if the reports explicitly locate the AVM in the tectum, perimesencephalic region, or brainstem.
2.  `"type of avm"`: The value must be a string describing the most specific location of the AVM mentioned in the reports (e.g., "Cerebellar AVM").
"""

In [11]:
PROMPT_2 = """Act as a specialized medical data extraction bot. Your goal is to read the following medical text and return a precise JSON object.

First, identify all anatomical terms used to describe the location of the AVM in the text (e.g., vermis, posterior fossa, cerebellar).

Second, determine if any of these identified locations are part of the tectum, perimesencephalic region, or brainstem.

Finally, construct a JSON object based on your analysis. The output must be ONLY the JSON, with no other commentary.

**JSON Structure:**
- `"tectal/perimesencephalic/brainstem AVM"`: (boolean) `true` if located in this region, `false` otherwise.
- `"type of avm"`: (string) The most specific location identified.
"""